# Purifying Selection Shapes the Architecture of Pigmentation GWAS

**Authors:** Tina Lasisi, Kirk Lohmueller

**Key Question:** Do common pigmentation-associated coding variants cluster in loss-of-function tolerant genes?

**Approach:**
1. Pull ALL pigmentation GWAS associations from GWAS Catalog API
2. Classify variants as coding vs regulatory based on functional annotation
3. Get constraint (LOEUF) for each gene from gnomAD
4. Test whether coding GWAS genes are more LoF-tolerant than regulatory GWAS genes

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import gzip
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
sns.set_palette("colorblind")

print("Packages loaded successfully")

## 2. Discover Pigmentation Traits in GWAS Catalog

First, we programmatically find all EFO trait IDs related to pigmentation.

In [ ]:
def get_all_traits(page_size=500):
    """Get all EFO traits from GWAS Catalog."""
    
    url = "https://www.ebi.ac.uk/gwas/rest/api/efoTraits"
    all_traits = []
    page = 0
    
    while True:
        print(f"Fetching trait page {page}...")
        response = requests.get(url, params={'size': page_size, 'page': page}, timeout=60)
        
        if response.status_code != 200:
            break
            
        data = response.json()
        
        if '_embedded' not in data or 'efoTraits' not in data['_embedded']:
            break
            
        traits = data['_embedded']['efoTraits']
        if not traits:
            break
            
        all_traits.extend(traits)
        
        if 'page' in data and data['page']['number'] >= data['page']['totalPages'] - 1:
            break
            
        page += 1
        time.sleep(0.3)
    
    return all_traits

print("="*70)
print("STEP 1: DISCOVERING PIGMENTATION TRAITS IN GWAS CATALOG")
print("="*70)

all_traits = get_all_traits()
print(f"\nTotal traits in GWAS Catalog: {len(all_traits)}")

# Convert to dataframe
df_traits = pd.DataFrame([
    {'efo_id': t['shortForm'], 'trait': t['trait']} 
    for t in all_traits
])

# Filter for pigmentation-related terms
pigment_keywords = [
    'pigment', 'skin color', 'skin colour', 'eye color', 'eye colour', 
    'hair color', 'hair colour', 'melanin', 'tanning', 'freckl', 
    'iris color', 'iris colour'
]

def is_pigmentation_trait(trait_name):
    trait_lower = trait_name.lower()
    # Exclude non-pigmentation traits that happen to contain keywords
    exclude = ['colorectal', 'retinitis pigmentosa', 'hemoglobin', 'skin cancer', 
               'skin disease', 'eye disease', 'hair loss', 'skin aging']
    if any(ex in trait_lower for ex in exclude):
        return False
    return any(kw in trait_lower for kw in pigment_keywords)

df_pigment_traits = df_traits[df_traits['trait'].apply(is_pigmentation_trait)]

print(f"\nPigmentation-related traits found: {len(df_pigment_traits)}")
print(df_pigment_traits.to_string(index=False))

## 3. Get GWAS Associations for Pigmentation Traits

In [ ]:
def get_trait_associations(efo_id, trait_name):
    """Get all associations for a trait from GWAS Catalog."""
    
    url = f"https://www.ebi.ac.uk/gwas/rest/api/efoTraits/{efo_id}/associations"
    
    try:
        response = requests.get(url, params={'size': 1000}, timeout=60)
        
        if response.status_code != 200:
            return []
        
        data = response.json()
        
        if '_embedded' not in data or 'associations' not in data['_embedded']:
            return []
        
        associations = data['_embedded']['associations']
        
        results = []
        for assoc in associations:
            pval = assoc.get('pvalue')
            
            for locus in assoc.get('loci', []):
                # Get SNP ID from risk allele
                snp_id = None
                risk_allele = None
                for ra in locus.get('strongestRiskAlleles', []):
                    risk_allele = ra.get('riskAlleleName', '')
                    if '-' in risk_allele:
                        snp_id = risk_allele.split('-')[0]
                
                # Get genes
                for g in locus.get('authorReportedGenes', []):
                    gene_name = g.get('geneName', '')
                    if gene_name and gene_name.lower() not in ['intergenic', 'nr', '', 'n/a']:
                        results.append({
                            'efo_id': efo_id,
                            'trait': trait_name,
                            'gene': gene_name.upper(),
                            'snp_id': snp_id,
                            'pvalue': pval
                        })
        
        return results
        
    except Exception as e:
        return []


# Use the validated pigmentation trait IDs
pigmentation_traits = {
    'EFO_0003924': 'hair color',
    'EFO_0003949': 'eye color',
    'EFO_0003963': 'freckles',
    'EFO_0009764': 'eye colour measurement',
    'OBA_VT0002095': 'skin pigmentation',
    'OBA_2045282': 'facial pigmentation',
    'EFO_0021835': 'melanin measurement',
    'EFO_0004795': 'skin sensitivity to sun',
}

print("="*70)
print("STEP 2: FETCHING GWAS ASSOCIATIONS")
print("="*70)

all_associations = []

for efo_id, trait_name in pigmentation_traits.items():
    print(f"  {efo_id} ({trait_name})...", end=" ")
    results = get_trait_associations(efo_id, trait_name)
    print(f"{len(results)} associations")
    all_associations.extend(results)
    time.sleep(0.5)

gwas_df = pd.DataFrame(all_associations)

print(f"\nTotal gene-trait associations: {len(gwas_df)}")
print(f"Unique genes: {gwas_df['gene'].nunique()}")
print(f"Unique SNPs: {gwas_df['snp_id'].nunique()}")

## 4. Get Functional Class for Each SNP

In [ ]:
def get_snp_functional_class(snp_id):
    """Get functional class for a SNP from GWAS Catalog."""
    
    if not snp_id or not snp_id.startswith('rs'):
        return None
    
    url = f"https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/{snp_id}"
    
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            data = response.json()
            return data.get('functionalClass')
        return None
    except:
        return None

print("="*70)
print("STEP 3: FETCHING SNP FUNCTIONAL CLASSES")
print("="*70)

unique_snps = gwas_df['snp_id'].dropna().unique()
print(f"Fetching functional class for {len(unique_snps)} unique SNPs...")
print("(This may take a few minutes)")

snp_func_classes = {}

for i, snp_id in enumerate(unique_snps):
    func_class = get_snp_functional_class(snp_id)
    if func_class:
        snp_func_classes[snp_id] = func_class
    
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(unique_snps)} SNPs...")
    
    time.sleep(0.15)

print(f"\nGot functional class for {len(snp_func_classes)} SNPs")

# Add to dataframe
gwas_df['functional_class'] = gwas_df['snp_id'].map(snp_func_classes)

print("\nFunctional class distribution:")
print(gwas_df['functional_class'].value_counts())

## 5. Classify Variants as Coding vs Regulatory

In [ ]:
def classify_variant(func_class):
    """Classify a variant as coding, regulatory, or other based on functional class."""
    
    if pd.isna(func_class):
        return 'unknown'
    
    func_lower = func_class.lower()
    
    # Coding variants - directly alter protein sequence
    if any(x in func_lower for x in ['missense', 'nonsense', 'frameshift', 'stop_gained', 
                                       'stop_lost', 'start_lost', 'splice_donor', 'splice_acceptor',
                                       'protein_altering']):
        return 'coding'
    
    # Regulatory variants - affect gene expression, not protein sequence
    if any(x in func_lower for x in ['intron', 'upstream', 'downstream', 'intergenic', 
                                       'regulatory', 'enhancer', '3_prime', '5_prime', 
                                       'non_coding', 'synonymous', 'utr', 'tf_binding']):
        return 'regulatory'
    
    return 'other'

gwas_df['variant_type'] = gwas_df['functional_class'].apply(classify_variant)

print("="*70)
print("STEP 4: VARIANT CLASSIFICATION")
print("="*70)

print("\nVariant-level classification:")
print(gwas_df['variant_type'].value_counts())

# Create gene-level summary
# A gene is "coding" if ANY of its variants are coding
# Otherwise "regulatory" if any are regulatory
# Otherwise "unknown"

def get_gene_variant_type(group):
    types = set(group['variant_type'])
    if 'coding' in types:
        return 'coding'
    elif 'regulatory' in types:
        return 'regulatory'
    elif 'other' in types:
        return 'other'
    else:
        return 'unknown'

gene_types = gwas_df.groupby('gene').apply(get_gene_variant_type).reset_index()
gene_types.columns = ['gene', 'variant_type']

print("\nGene-level classification:")
print(gene_types['variant_type'].value_counts())

print("\nCODING genes:")
coding_genes = gene_types[gene_types['variant_type'] == 'coding']['gene'].tolist()
print(coding_genes)

print(f"\nREGULATORY genes: {len(gene_types[gene_types['variant_type'] == 'regulatory'])} genes")

## 6. Get Constraint (LOEUF) from gnomAD

In [ ]:
def get_gnomad_constraint():
    """Download and load gnomAD constraint data."""
    
    # Try local file first
    local_files = [
        '../data/gnomad.v2.1.1.lof_metrics.by_gene.txt.bgz',
        '../data/gnomad_constraint.txt.bgz',
    ]
    
    for f in local_files:
        try:
            with gzip.open(f, 'rt') as fh:
                df = pd.read_csv(fh, sep='\t')
            print(f"Loaded constraint data from {f}")
            return df
        except:
            continue
    
    # Download if not found
    print("Downloading gnomAD constraint data...")
    url = "https://storage.googleapis.com/gcp-public-data--gnomad/release/2.1.1/constraint/gnomad.v2.1.1.lof_metrics.by_gene.txt.bgz"
    
    try:
        response = requests.get(url, timeout=120)
        if response.status_code == 200:
            with open('../data/gnomad_constraint.txt.bgz', 'wb') as f:
                f.write(response.content)
            with gzip.open('../data/gnomad_constraint.txt.bgz', 'rt') as fh:
                df = pd.read_csv(fh, sep='\t')
            print("Downloaded gnomAD constraint data")
            return df
    except Exception as e:
        print(f"Error: {e}")
    
    return None

print("="*70)
print("STEP 5: LOADING gnomAD CONSTRAINT DATA")
print("="*70)

gnomad_df = get_gnomad_constraint()

if gnomad_df is not None:
    # Get canonical transcripts only
    if 'canonical' in gnomad_df.columns:
        gnomad_df = gnomad_df[gnomad_df['canonical'] == True]
    
    # Create lookup dict
    gnomad_df['gene_upper'] = gnomad_df['gene'].str.upper()
    loeuf_lookup = gnomad_df.set_index('gene_upper')['oe_lof_upper'].to_dict()
    
    print(f"Loaded LOEUF for {len(loeuf_lookup)} genes")
    
    # Add LOEUF to gene_types
    gene_types['LOEUF'] = gene_types['gene'].map(loeuf_lookup)
    
    print(f"\nGenes with LOEUF data: {gene_types['LOEUF'].notna().sum()}/{len(gene_types)}")

## 7. Statistical Analysis

In [ ]:
print("="*70)
print("STEP 6: STATISTICAL ANALYSIS")
print("="*70)

# Filter to genes with known variant type and LOEUF
df_analysis = gene_types[
    (gene_types['variant_type'].isin(['coding', 'regulatory'])) & 
    (gene_types['LOEUF'].notna())
].copy()

print(f"\nAnalyzing {len(df_analysis)} genes with known variant type and LOEUF")

coding = df_analysis[df_analysis['variant_type'] == 'coding']['LOEUF'].values
regulatory = df_analysis[df_analysis['variant_type'] == 'regulatory']['LOEUF'].values

print(f"\nCODING GWAS genes (n={len(coding)}):")
print(f"  Median LOEUF: {np.median(coding):.2f}")
print(f"  Mean LOEUF: {np.mean(coding):.2f}")
print(f"  IQR: [{np.percentile(coding, 25):.2f} - {np.percentile(coding, 75):.2f}]")

print(f"\nREGULATORY GWAS genes (n={len(regulatory)}):")
print(f"  Median LOEUF: {np.median(regulatory):.2f}")
print(f"  Mean LOEUF: {np.mean(regulatory):.2f}")
print(f"  IQR: [{np.percentile(regulatory, 25):.2f} - {np.percentile(regulatory, 75):.2f}]")

# Mann-Whitney U test
u_stat, p_two = stats.mannwhitneyu(coding, regulatory, alternative='two-sided')
_, p_one = stats.mannwhitneyu(coding, regulatory, alternative='greater')

print(f"\n" + "-"*50)
print("MANN-WHITNEY U TEST")
print("-"*50)
print(f"U statistic: {u_stat:.0f}")
print(f"Two-tailed p-value: {p_two:.4f}")
print(f"One-tailed p-value (coding > regulatory): {p_one:.4f}")

# Effect size (rank-biserial correlation)
n1, n2 = len(coding), len(regulatory)
r = 1 - (2 * u_stat) / (n1 * n2)
print(f"Effect size (rank-biserial r): {r:.2f}")

## 8. Figure

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

# Prepare data
categories = ['Coding\nGWAS', 'Regulatory\nGWAS']
data_lists = [coding, regulatory]
colors = ['#9b59b6', '#f39c12']

# Create boxplots with jittered points
for i, (data, color) in enumerate(zip(data_lists, colors)):
    bp = ax.boxplot([data], positions=[i+1], widths=0.5, patch_artist=True)
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.7)
    bp['medians'][0].set_color('black')
    bp['medians'][0].set_linewidth(2)
    
    # Jittered points
    jitter = np.random.normal(0, 0.06, len(data))
    ax.scatter([i+1] * len(data) + jitter, data, c=color, alpha=0.5, s=20, zorder=3)

# Add threshold lines
ax.axhline(y=0.35, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(2.55, 0.35, 'Constrained\n(LOEUF < 0.35)', va='center', fontsize=8, color='red', alpha=0.7)
ax.text(2.55, 1.0, 'Tolerant\n(LOEUF > 1.0)', va='center', fontsize=8, color='gray', alpha=0.7)

# Significance annotation
if p_one < 0.001:
    sig_text = '***'
elif p_one < 0.01:
    sig_text = '**'
elif p_one < 0.05:
    sig_text = '*'
else:
    sig_text = f'p = {p_one:.3f}'

y_max = max(max(coding), max(regulatory))
ax.annotate(sig_text, xy=(1.5, y_max + 0.15), ha='center', fontsize=12)
ax.plot([1, 1, 2, 2], [y_max + 0.05, y_max + 0.1, y_max + 0.1, y_max + 0.05], 'k-', linewidth=1)

ax.set_xticks([1, 2])
ax.set_xticklabels(categories)
ax.set_ylabel('LOEUF (gnomAD v2.1.1)\n← More constrained | More tolerant →')
ax.set_title(f'Constraint by GWAS Variant Type\n(n={len(coding)} coding, n={len(regulatory)} regulatory genes)')
ax.set_xlim(0.4, 3.0)
ax.set_ylim(0, y_max + 0.4)

plt.tight_layout()
plt.savefig('figure_pigmentation_constraint.png', dpi=300, bbox_inches='tight')
plt.savefig('figure_pigmentation_constraint.pdf', bbox_inches='tight')
plt.show()

print("Figure saved!")

## 9. Summary Table

In [ ]:
# Create summary of coding genes
print("="*70)
print("CODING GWAS GENES")
print("="*70)

coding_detail = df_analysis[df_analysis['variant_type'] == 'coding'].sort_values('LOEUF', ascending=False)
print(coding_detail[['gene', 'LOEUF']].to_string(index=False))

# Save full results
df_analysis.to_csv('pigmentation_gwas_constraint.csv', index=False)
print(f"\nSaved full results to: pigmentation_gwas_constraint.csv")

## 10. Data Sources

| Data | Source | Access Method |
|------|--------|---------------|
| GWAS associations | NHGRI-EBI GWAS Catalog | REST API |
| Variant functional class | GWAS Catalog SNP annotations | REST API |
| Gene constraint (LOEUF) | gnomAD v2.1.1 | Direct download |

All data was retrieved programmatically - no hardcoded gene lists.